In [ ]:
#Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sys

#Root
project_root = Path.cwd().parent
sys.path.append(str(project_root))

#Utils
from src.utils.db import get_engine
from src.utils.logger import setup_logger

logger = setup_logger(__name__)
engine = get_engine()

logger.info("Environment ready")

In [ ]:
query = """
    SELECT
        date,
        COUNT(*) as num_transactions,
        ROUND(SUM(unit_sales),0) as total_sales,
        ROUND(AVG(unit_sales),2) as avg_sales,
        COUNT(DISTINCT store_nbr) as active_stores,
        COUNT(DISTINCT item_nbr) as unique_items
    FROM train
    GROUP BY date
    ORDER BY date;
"""

logger.info("Loading sales data...")
daily_sales = pd.read_sql(query, engine)
daily_sales['date'] = pd.to_datetime(daily_sales['date'])

logger.info(f"Loaded {len(daily_sales)} days of data")

In [ ]:

#Daily sales statistics
print("\nDaily Sales Statistics:\n")
print(daily_sales[['total_sales','num_transactions','avg_sales']].describe())

#New features
daily_sales['year'] = daily_sales['date'].dt.year
daily_sales['month'] = daily_sales['date'].dt.month
daily_sales['day_of_week'] = daily_sales['date'].dt.day_of_week
daily_sales['day_name'] = daily_sales['date'].dt.day_name()
daily_sales['week_of_year'] = daily_sales['date'].dt.isocalendar().week

print("\nDaily Sales Sample:\n")
display(daily_sales.head(10))

print("\nDate range by year:\n")
print(daily_sales.groupby('year')['date'].agg(['min','max','count']))



In [ ]:
#Sales trend

fig = go.Figure()

fig.add_trace(
    go.Scatter(x=daily_sales['date'],y=daily_sales['total_sales'],mode='lines', name ='Daily Sales', line=dict(color = '#1f77b4',width = 1))
    )

daily_sales['ma_30'] = daily_sales['total_sales'].rolling(window=30).mean()

fig.add_trace(
    go.Scatter(x=daily_sales['date'],y=daily_sales['ma_30'],mode='lines', name ='30-day Average', line=dict(color = '#ff7f0e',width = 1))
    )

fig.update_layout(
    title='Daily Sales Trend (2013 - 2017)',
    xaxis_title = 'Date',
    yaxis_title = 'Total Sales',
    hovermode = 'x unified',
    height = 500

)

fig.show()

In [ ]:
#Daily sales vs oil cost
oil = pd.read_sql("SELECT * FROM oil", engine)

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(x=daily_sales['date'],y=daily_sales['ma_30'],mode='lines', name ='Sales 30-Days Avg', line=dict(color = '#1f77b4',width = 1)),
    secondary_y=False
    )

fig.add_trace(
    go.Scatter(x=oil['date'],y=oil['dcoilwtico'],mode='lines', name ='Oil Cost', line=dict(color = '#ff7f0e',width = 1)),
    secondary_y=True
    )

fig.update_layout(
    title='Daily Sales vs Oil Cost Trend (2013 - 2017)',
    xaxis_title = 'Date',
    hovermode = 'x unified',
    height = 500

)

fig.update_yaxes(title_text="<b>Total Sales</b> (Unity)", secondary_y=False)
fig.update_yaxes(title_text="<b>Oil Price</b> (USD)", secondary_y=True)
fig.show()




In [ ]:
#Year Comparison

yearly_sales = daily_sales.groupby('year').agg({'total_sales': 'sum',
                                                'num_transactions': 'sum',
                                                'date':'count'}).round(0)

yearly_sales.columns = ['Total Sales', 'Total Transactions','# Days']
yearly_sales['Avg Daily Sales'] = (yearly_sales['Total Sales'] / yearly_sales['# Days']).round(0)

print("\nAnual year sales summary:")

display(yearly_sales)

fig = make_subplots(
    rows = 1, cols= 2, subplot_titles=('Total Sales by Year', 'Average Daily Sales by Year')
)


fig.add_trace(
    go.Bar(x=yearly_sales.index, y=yearly_sales['Total Sales'], marker_color='#1f77b4'),
    row = 1, col = 1
)

fig.add_trace(
    go.Bar(x=yearly_sales.index, y=yearly_sales['Avg Daily Sales'], marker_color = '#ff7f0e'),
    row = 1, col = 2
)

fig.update_layout(height=400, showlegend=False,
                  xaxis_title='Year',yaxis_title='Total Sales',
                  xaxis2_title='Year',yaxis2_title='Daily Sales',
                  )

fig.show()

In [ ]:
#Month sales

monthly_pattern = daily_sales.groupby('month').agg(
    {
        'total_sales': 'mean',
        'num_transactions': 'mean',

    }
).round(0)

monthly_pattern.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
 

#Identifying best months
best_month = monthly_pattern['total_sales'].idxmax()
worst_month = monthly_pattern['total_sales'].idxmin()

print(f"\n Best month: {best_month}")
print(f"\n Worst month: {worst_month}")


fig = go.Figure()

fig.add_trace(
    go.Bar(x=monthly_pattern.index,y=monthly_pattern['total_sales'],marker_color='#1f77b4',text=monthly_pattern['total_sales'],textposition='outside'
))

fig.update_layout(
    title='Average Sales by Month (Seasonality)', xaxis_title = 'Month', yaxis_title = 'Average Month Sales'
)

fig.show()




In [ ]:
dow_sales = daily_sales.groupby('day_name').agg(
    {'total_sales':'mean',
     'num_transactions':'mean',
    }
)

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_sales = dow_sales.reindex(day_order)
display(dow_sales.head(7))

fig = go.Figure()

fig.add_trace(

    go.Bar(x=dow_sales.index,y=dow_sales['total_sales'], text=dow_sales['total_sales'],textposition='outside',)
)

fig.update_layout(height = 500, title = 'Average Daily Sales by Day of Week', xaxis_title = 'Day',yaxis_title = 'Average sales')
fig.update_traces(texttemplate='%{text:,.2f}')


fig.show()

# Calculate percentage difference
avg_sales = dow_sales['total_sales'].mean()
dow_sales['% vs Average'] = ((dow_sales['total_sales'] / avg_sales - 1) * 100).round(1)

print("\nPerformance vs Average:")
display(dow_sales[['total_sales', '% vs Average']])

In [ ]:
#Holidays
holidays = pd.read_sql("SELECT * FROM holidays_events ORDER BY date;",engine)

holidays.head()

print(f"\nTotal Holidays in dataset: {len(holidays)}")
print("\nHolidays by type:")
print(holidays['type'].value_counts())

holidays['date'] = pd.to_datetime(holidays['date'])

daily_sales_holidays = daily_sales.merge(
    holidays[['date', 'type', 'description']], 
    on='date', 
    how='left'
)

daily_sales_holidays['is_holiday'] = daily_sales_holidays['type'].notna()

holiday_comparison = daily_sales_holidays.groupby('is_holiday').agg({
    'total_sales':'mean',
}).round()


holiday_comparison.index = ['Regular Days', 'Holidays']

print(f"\nRegular days vs Holiday Sales:")
display(holiday_comparison)

impact = ((holiday_comparison.loc['Holidays', 'total_sales'] / 
           holiday_comparison.loc['Regular Days', 'total_sales'] - 1) * 100)

print(f"\n Holiday Impact: {impact:+.1f}% vs regular days")

print(f"\nHoliday Impact")


fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=holiday_comparison.index, y = holiday_comparison['total_sales'],text=holiday_comparison['total_sales']
    )
)

fig.update_layout(height = 500, title = 'Avg sales in Regular Days vs Holidays', xaxis_title = 'Day Type',yaxis_title = 'Average sales')
fig.update_traces(texttemplate='%{text:,.2f}')


In [ ]:
#December sales
december_sales = daily_sales[daily_sales['month'] == 12].copy()
december_sales['day'] = december_sales['date'].dt.day

print("\nDecember Statistics:")
print(december_sales.groupby('year')['total_sales'].agg(['mean', 'max', 'min']))

fig = go.Figure()

for year in december_sales['year'].unique():
    year_data = december_sales[december_sales['year'] == year]
    fig.add_trace(go.Scatter(
        x=year_data['day'],
        y=year_data['total_sales'],
        mode='lines+markers',
        name=str(year)
    ))

    fig.update_layout(
    title='December Daily Sales by Year',
    xaxis_title='Day of December',
    yaxis_title='Daily Sales',
    hovermode='x unified',
    height=500
)

fig.show()



Temporal Findings:

1. Overall Trend:
    - Sales show growth from 2013 to 2017.
    - The 30-day moving average effectively smooths out daily fluctuations.
    - December is the strongest month for sales, while February is the weakest.

2. Weekly Patterns:
    - Sundays show the hightest sales volume.
    - A clear distinction exists between weekends and weekdays, with weekends significantly outperforming the daily average.

3. Holiday Impact:
    - Holidays show 11.8% higher sales compared to regular days.
    - Christmas season drives significant revenue.

4. Oil related:
    - There appears to be an inverse relationship (negative correlation) between oil prices and daily sales. 
